# 01 — End-to-End Lakehouse Smoke Test

## Purpose

Validate the operational health of the complete Revenue Leakage and Customer 360 Lakehouse.

The smoke test confirms that all expected Bronze, Silver, and Gold Delta tables are available, contain the expected number of records, preserve business-key uniqueness, and reconcile across pipeline layers.

## Pipeline Coverage

### Bronze

- Customer events
- Subscription events
- Invoice events
- Payment events

### Silver

- Customers
- Subscriptions
- Invoices
- Payments

### Gold

- Customer 360
- Revenue Leakage
- Payment Recovery
- Executive KPIs

## Test Behavior

This notebook performs read-only validation. It does not modify, overwrite, merge, or delete any pipeline data.

In [0]:
from pyspark.sql import functions as F


CATALOG = "workspace"

BRONZE_SCHEMA = "revenue_leakage_bronze"
SILVER_SCHEMA = "revenue_leakage_silver"
GOLD_SCHEMA = "revenue_leakage_gold"


TABLE_CONFIG = [
    {
        "layer": "Bronze",
        "domain": "Customers",
        "table_name": (
            f"{CATALOG}.{BRONZE_SCHEMA}.customer_events"
        ),
        "expected_row_count": 5_500,
    },
    {
        "layer": "Bronze",
        "domain": "Subscriptions",
        "table_name": (
            f"{CATALOG}.{BRONZE_SCHEMA}.subscription_events"
        ),
        "expected_row_count": 6_550,
    },
    {
        "layer": "Bronze",
        "domain": "Invoices",
        "table_name": (
            f"{CATALOG}.{BRONZE_SCHEMA}.invoice_events"
        ),
        "expected_row_count": 26_925,
    },
    {
        "layer": "Bronze",
        "domain": "Payments",
        "table_name": (
            f"{CATALOG}.{BRONZE_SCHEMA}.payment_events"
        ),
        "expected_row_count": 30_232,
    },
    {
        "layer": "Silver",
        "domain": "Customers",
        "table_name": (
            f"{CATALOG}.{SILVER_SCHEMA}.customers"
        ),
        "expected_row_count": 5_150,
    },
    {
        "layer": "Silver",
        "domain": "Subscriptions",
        "table_name": (
            f"{CATALOG}.{SILVER_SCHEMA}.subscriptions"
        ),
        "expected_row_count": 6_200,
    },
    {
        "layer": "Silver",
        "domain": "Invoices",
        "table_name": (
            f"{CATALOG}.{SILVER_SCHEMA}.invoices"
        ),
        "expected_row_count": 26_699,
    },
    {
        "layer": "Silver",
        "domain": "Payments",
        "table_name": (
            f"{CATALOG}.{SILVER_SCHEMA}.payments"
        ),
        "expected_row_count": 29_972,
    },
    {
        "layer": "Gold",
        "domain": "Customer 360",
        "table_name": (
            f"{CATALOG}.{GOLD_SCHEMA}.customer_360"
        ),
        "expected_row_count": 5_150,
    },
    {
        "layer": "Gold",
        "domain": "Revenue Leakage",
        "table_name": (
            f"{CATALOG}.{GOLD_SCHEMA}.revenue_leakage"
        ),
        "expected_row_count": 26_699,
    },
    {
        "layer": "Gold",
        "domain": "Payment Recovery",
        "table_name": (
            f"{CATALOG}.{GOLD_SCHEMA}.payment_recovery"
        ),
        "expected_row_count": 25_748,
    },
    {
        "layer": "Gold",
        "domain": "Executive KPIs",
        "table_name": (
            f"{CATALOG}.{GOLD_SCHEMA}.executive_kpis"
        ),
        "expected_row_count": 1,
    },
]


missing_tables = [
    table_config["table_name"]
    for table_config in TABLE_CONFIG
    if not spark.catalog.tableExists(
        table_config["table_name"]
    )
]


assert not missing_tables, (
    "Required Lakehouse tables are missing: "
    f"{missing_tables}"
)


table_inventory_results = []


for table_config in TABLE_CONFIG:
    layer_name = table_config["layer"]
    domain_name = table_config["domain"]
    table_name = table_config["table_name"]
    expected_row_count = (
        table_config["expected_row_count"]
    )

    actual_row_count = (
        spark.table(table_name).count()
    )

    table_detail_row = (
        spark.sql(
            f"DESCRIBE DETAIL {table_name}"
        )
        .select("format")
        .first()
    )

    table_format = (
        table_detail_row["format"]
    )

    row_count_status = (
        "PASS"
        if actual_row_count == expected_row_count
        else "FAIL"
    )

    format_status = (
        "PASS"
        if table_format.lower() == "delta"
        else "FAIL"
    )

    overall_status = (
        "PASS"
        if (
            row_count_status == "PASS"
            and format_status == "PASS"
        )
        else "FAIL"
    )

    table_inventory_results.append(
        (
            layer_name,
            domain_name,
            table_name,
            expected_row_count,
            actual_row_count,
            table_format,
            row_count_status,
            format_status,
            overall_status,
        )
    )


smoke_test_inventory_df = (
    spark.createDataFrame(
        table_inventory_results,
        [
            "layer",
            "domain",
            "table_name",
            "expected_row_count",
            "actual_row_count",
            "table_format",
            "row_count_status",
            "format_status",
            "overall_status",
        ],
    )
)


failed_inventory_test_count = (
    smoke_test_inventory_df
    .where(
        F.col("overall_status") != "PASS"
    )
    .count()
)


assert failed_inventory_test_count == 0, (
    "One or more Lakehouse inventory tests failed."
)


print(
    "Required Lakehouse tables: "
    f"{len(TABLE_CONFIG):,}"
)

print(
    "Missing tables: "
    f"{len(missing_tables):,}"
)

print(
    "Failed inventory tests: "
    f"{failed_inventory_test_count:,}"
)

print(
    "All required Bronze, Silver, and Gold "
    "Delta tables are available."
)


display(
    smoke_test_inventory_df
    .orderBy(
        F.when(
            F.col("layer") == "Bronze",
            1,
        )
        .when(
            F.col("layer") == "Silver",
            2,
        )
        .otherwise(3),
        "domain",
    )
)

## 2. Validate Business Keys and Referential Integrity

Validate required business-key completeness and uniqueness across Bronze, Silver, and Gold.

Confirm that subscriptions, invoices, payments, Customer 360 records, Revenue Leakage records, and Payment Recovery journeys preserve valid references to their parent entities.

In [0]:
# Validate business-key uniqueness and cross-table references.

KEY_TEST_CONFIG = [
    (
        "Bronze",
        "Customers",
        f"{CATALOG}.{BRONZE_SCHEMA}.customer_events",
        "_record_hash",
        5_500,
    ),
    (
        "Bronze",
        "Subscriptions",
        f"{CATALOG}.{BRONZE_SCHEMA}.subscription_events",
        "_record_hash",
        6_550,
    ),
    (
        "Bronze",
        "Invoices",
        f"{CATALOG}.{BRONZE_SCHEMA}.invoice_events",
        "_record_hash",
        26_925,
    ),
    (
        "Bronze",
        "Payments",
        f"{CATALOG}.{BRONZE_SCHEMA}.payment_events",
        "_record_hash",
        30_232,
    ),
    (
        "Silver",
        "Customers",
        f"{CATALOG}.{SILVER_SCHEMA}.customers",
        "customer_id",
        5_150,
    ),
    (
        "Silver",
        "Subscriptions",
        f"{CATALOG}.{SILVER_SCHEMA}.subscriptions",
        "subscription_id",
        6_200,
    ),
    (
        "Silver",
        "Invoices",
        f"{CATALOG}.{SILVER_SCHEMA}.invoices",
        "invoice_id",
        26_699,
    ),
    (
        "Silver",
        "Payments",
        f"{CATALOG}.{SILVER_SCHEMA}.payments",
        "payment_id",
        29_972,
    ),
    (
        "Silver",
        "Payment Transactions",
        f"{CATALOG}.{SILVER_SCHEMA}.payments",
        "provider_transaction_id",
        29_972,
    ),
    (
        "Gold",
        "Customer 360",
        f"{CATALOG}.{GOLD_SCHEMA}.customer_360",
        "customer_id",
        5_150,
    ),
    (
        "Gold",
        "Revenue Leakage",
        f"{CATALOG}.{GOLD_SCHEMA}.revenue_leakage",
        "invoice_id",
        26_699,
    ),
    (
        "Gold",
        "Payment Recovery",
        f"{CATALOG}.{GOLD_SCHEMA}.payment_recovery",
        "invoice_id",
        25_748,
    ),
    (
        "Gold",
        "Executive KPIs",
        f"{CATALOG}.{GOLD_SCHEMA}.executive_kpis",
        "executive_kpi_snapshot_key",
        1,
    ),
]


key_test_results = []


for (
    layer_name,
    domain_name,
    table_name,
    key_column,
    expected_distinct_count,
) in KEY_TEST_CONFIG:
    source_df = spark.table(table_name)

    source_row_count = source_df.count()

    null_key_count = (
        source_df
        .where(
            F.col(key_column).isNull()
            | (
                F.trim(
                    F.col(key_column).cast("string")
                )
                == ""
            )
        )
        .count()
    )

    distinct_key_count = (
        source_df
        .select(key_column)
        .where(
            F.col(key_column).isNotNull()
        )
        .distinct()
        .count()
    )

    duplicate_key_count = (
        source_df
        .groupBy(key_column)
        .count()
        .where(
            F.col(key_column).isNotNull()
            & (F.col("count") > 1)
        )
        .count()
    )

    key_test_status = (
        "PASS"
        if (
            null_key_count == 0
            and distinct_key_count
            == expected_distinct_count
            and duplicate_key_count == 0
        )
        else "FAIL"
    )

    key_test_results.append(
        (
            layer_name,
            domain_name,
            table_name,
            key_column,
            source_row_count,
            expected_distinct_count,
            distinct_key_count,
            null_key_count,
            duplicate_key_count,
            key_test_status,
        )
    )


key_integrity_results_df = (
    spark.createDataFrame(
        key_test_results,
        [
            "layer",
            "domain",
            "table_name",
            "key_column",
            "row_count",
            "expected_distinct_key_count",
            "actual_distinct_key_count",
            "null_key_count",
            "duplicate_key_count",
            "test_status",
        ],
    )
)


silver_customers_df = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.customers"
)

silver_subscriptions_df = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.subscriptions"
)

silver_invoices_df = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.invoices"
)

silver_payments_df = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.payments"
)

gold_customer_360_df = spark.table(
    f"{CATALOG}.{GOLD_SCHEMA}.customer_360"
)

gold_revenue_leakage_df = spark.table(
    f"{CATALOG}.{GOLD_SCHEMA}.revenue_leakage"
)

gold_payment_recovery_df = spark.table(
    f"{CATALOG}.{GOLD_SCHEMA}.payment_recovery"
)


REFERENCE_TEST_CONFIG = [
    (
        "Silver subscriptions to customers",
        silver_subscriptions_df,
        "customer_id",
        silver_customers_df,
        "customer_id",
    ),
    (
        "Silver invoices to customers",
        silver_invoices_df,
        "customer_id",
        silver_customers_df,
        "customer_id",
    ),
    (
        "Silver invoices to subscriptions",
        silver_invoices_df,
        "subscription_id",
        silver_subscriptions_df,
        "subscription_id",
    ),
    (
        "Silver payments to customers",
        silver_payments_df,
        "customer_id",
        silver_customers_df,
        "customer_id",
    ),
    (
        "Silver payments to subscriptions",
        silver_payments_df,
        "subscription_id",
        silver_subscriptions_df,
        "subscription_id",
    ),
    (
        "Silver payments to invoices",
        silver_payments_df,
        "invoice_id",
        silver_invoices_df,
        "invoice_id",
    ),
    (
        "Gold Customer 360 to Silver customers",
        gold_customer_360_df,
        "customer_id",
        silver_customers_df,
        "customer_id",
    ),
    (
        "Gold Revenue Leakage to Customer 360",
        gold_revenue_leakage_df,
        "customer_id",
        gold_customer_360_df,
        "customer_id",
    ),
    (
        "Gold Revenue Leakage to Silver invoices",
        gold_revenue_leakage_df,
        "invoice_id",
        silver_invoices_df,
        "invoice_id",
    ),
    (
        "Gold Payment Recovery to Revenue Leakage",
        gold_payment_recovery_df,
        "invoice_id",
        gold_revenue_leakage_df,
        "invoice_id",
    ),
]


reference_test_results = []


for (
    test_name,
    child_df,
    child_key,
    parent_df,
    parent_key,
) in REFERENCE_TEST_CONFIG:
    orphan_reference_count = (
        child_df
        .select(
            F.col(child_key).alias(
                "reference_key"
            )
        )
        .where(
            F.col("reference_key").isNotNull()
        )
        .distinct()
        .join(
            parent_df
            .select(
                F.col(parent_key).alias(
                    "reference_key"
                )
            )
            .where(
                F.col("reference_key").isNotNull()
            )
            .distinct(),
            on="reference_key",
            how="left_anti",
        )
        .count()
    )

    reference_test_status = (
        "PASS"
        if orphan_reference_count == 0
        else "FAIL"
    )

    reference_test_results.append(
        (
            test_name,
            child_key,
            parent_key,
            orphan_reference_count,
            reference_test_status,
        )
    )


reference_integrity_results_df = (
    spark.createDataFrame(
        reference_test_results,
        [
            "reference_test",
            "child_key",
            "parent_key",
            "orphan_reference_count",
            "test_status",
        ],
    )
)


failed_key_test_count = (
    key_integrity_results_df
    .where(
        F.col("test_status") != "PASS"
    )
    .count()
)

failed_reference_test_count = (
    reference_integrity_results_df
    .where(
        F.col("test_status") != "PASS"
    )
    .count()
)


assert failed_key_test_count == 0, (
    "One or more business-key tests failed."
)

assert failed_reference_test_count == 0, (
    "One or more reference-integrity tests failed."
)


print(
    "Business-key tests: "
    f"{len(KEY_TEST_CONFIG):,}"
)

print(
    "Failed business-key tests: "
    f"{failed_key_test_count:,}"
)

print(
    "Reference-integrity tests: "
    f"{len(REFERENCE_TEST_CONFIG):,}"
)

print(
    "Failed reference-integrity tests: "
    f"{failed_reference_test_count:,}"
)

print(
    "All business-key and reference-integrity "
    "tests passed."
)


display(
    key_integrity_results_df
    .orderBy(
        "layer",
        "domain",
    )
)

display(
    reference_integrity_results_df
    .orderBy(
        "reference_test"
    )
)

## 3. Reconcile Silver, Gold, and Executive Metrics

Reconcile core record counts and financial metrics across the validated Silver tables, Gold analytical models, and the Executive KPI snapshot.

Every target metric must match its authoritative upstream source with a zero difference.

In [0]:
# Reconcile core business and financial metrics across pipeline layers.

gold_executive_kpis_df = spark.table(
    f"{CATALOG}.{GOLD_SCHEMA}.executive_kpis"
)


latest_executive_kpi_row = (
    gold_executive_kpis_df
    .orderBy(
        F.col(
            "analytics_snapshot_date"
        ).desc()
    )
    .first()
)


silver_invoice_metrics = (
    silver_invoices_df
    .agg(
        F.count("*").alias(
            "invoice_count"
        ),
        F.round(
            F.sum("invoice_total_amount"),
            2,
        ).alias(
            "invoice_total_amount"
        ),
        F.round(
            F.sum("amount_paid"),
            2,
        ).alias(
            "amount_paid"
        ),
        F.round(
            F.sum("outstanding_amount"),
            2,
        ).alias(
            "outstanding_amount"
        ),
        F.round(
            F.sum("voided_amount"),
            2,
        ).alias(
            "voided_amount"
        ),
    )
    .first()
)


gold_leakage_metrics = (
    gold_revenue_leakage_df
    .agg(
        F.count("*").alias(
            "invoice_count"
        ),
        F.round(
            F.sum("invoice_total_amount"),
            2,
        ).alias(
            "invoice_total_amount"
        ),
        F.round(
            F.sum("amount_paid"),
            2,
        ).alias(
            "amount_paid"
        ),
        F.round(
            F.sum("outstanding_amount"),
            2,
        ).alias(
            "outstanding_amount"
        ),
        F.round(
            F.sum("voided_amount"),
            2,
        ).alias(
            "voided_amount"
        ),
        F.round(
            F.sum(
                "total_revenue_exposure_amount"
            ),
            2,
        ).alias(
            "total_revenue_exposure_amount"
        ),
    )
    .first()
)


silver_payment_metrics = (
    silver_payments_df
    .agg(
        F.count("*").alias(
            "payment_attempt_count"
        ),
        F.round(
            F.sum("transaction_amount"),
            2,
        ).alias(
            "payment_attempt_amount"
        ),
        F.round(
            F.sum("settled_amount"),
            2,
        ).alias(
            "settled_amount"
        ),
    )
    .first()
)


silver_payment_journey_count = (
    silver_payments_df
    .select("invoice_id")
    .distinct()
    .count()
)


gold_recovery_metrics = (
    gold_payment_recovery_df
    .agg(
        F.count("*").alias(
            "payment_journey_count"
        ),
        F.round(
            F.sum("payment_attempt_amount"),
            2,
        ).alias(
            "payment_attempt_amount"
        ),
        F.round(
            F.sum("settled_amount"),
            2,
        ).alias(
            "settled_amount"
        ),
        F.round(
            F.sum("recovered_amount"),
            2,
        ).alias(
            "recovered_amount"
        ),
        F.round(
            F.sum("unrecovered_amount"),
            2,
        ).alias(
            "unrecovered_amount"
        ),
        F.round(
            F.sum("pending_collection_amount"),
            2,
        ).alias(
            "pending_collection_amount"
        ),
    )
    .first()
)


silver_customer_count = (
    silver_customers_df.count()
)

gold_customer_count = (
    gold_customer_360_df.count()
)


gold_customer_metrics = (
    gold_customer_360_df
    .agg(
        F.round(
            F.sum("current_mrr"),
            2,
        ).alias(
            "current_mrr"
        ),
        F.round(
            F.sum("current_arr"),
            2,
        ).alias(
            "current_arr"
        ),
    )
    .first()
)


reconciliation_results = []


def add_reconciliation_result(
    metric_name,
    source_layer,
    target_layer,
    source_value,
    target_value,
    tolerance=0.01,
):
    numeric_source_value = float(
        source_value or 0
    )

    numeric_target_value = float(
        target_value or 0
    )

    metric_difference = round(
        numeric_source_value
        - numeric_target_value,
        2,
    )

    reconciliation_status = (
        "PASS"
        if abs(metric_difference) <= tolerance
        else "FAIL"
    )

    reconciliation_results.append(
        (
            metric_name,
            source_layer,
            target_layer,
            numeric_source_value,
            numeric_target_value,
            metric_difference,
            reconciliation_status,
        )
    )


add_reconciliation_result(
    "customer_count",
    "Silver customers",
    "Gold Customer 360",
    silver_customer_count,
    gold_customer_count,
    0,
)

add_reconciliation_result(
    "invoice_count",
    "Silver invoices",
    "Gold Revenue Leakage",
    silver_invoice_metrics["invoice_count"],
    gold_leakage_metrics["invoice_count"],
    0,
)

add_reconciliation_result(
    "payment_journey_count",
    "Silver payment invoice journeys",
    "Gold Payment Recovery",
    silver_payment_journey_count,
    gold_recovery_metrics[
        "payment_journey_count"
    ],
    0,
)

add_reconciliation_result(
    "invoice_total_amount",
    "Silver invoices",
    "Gold Revenue Leakage",
    silver_invoice_metrics[
        "invoice_total_amount"
    ],
    gold_leakage_metrics[
        "invoice_total_amount"
    ],
)

add_reconciliation_result(
    "collected_amount",
    "Silver invoices",
    "Gold Revenue Leakage",
    silver_invoice_metrics["amount_paid"],
    gold_leakage_metrics["amount_paid"],
)

add_reconciliation_result(
    "outstanding_amount",
    "Silver invoices",
    "Gold Revenue Leakage",
    silver_invoice_metrics[
        "outstanding_amount"
    ],
    gold_leakage_metrics[
        "outstanding_amount"
    ],
)

add_reconciliation_result(
    "voided_amount",
    "Silver invoices",
    "Gold Revenue Leakage",
    silver_invoice_metrics["voided_amount"],
    gold_leakage_metrics["voided_amount"],
)

add_reconciliation_result(
    "payment_attempt_count",
    "Silver payments",
    "Executive KPIs",
    silver_payment_metrics[
        "payment_attempt_count"
    ],
    latest_executive_kpi_row[
        "payment_attempt_count"
    ],
    0,
)

add_reconciliation_result(
    "payment_attempt_amount",
    "Silver payments",
    "Gold Payment Recovery",
    silver_payment_metrics[
        "payment_attempt_amount"
    ],
    gold_recovery_metrics[
        "payment_attempt_amount"
    ],
)

add_reconciliation_result(
    "settled_amount",
    "Silver payments",
    "Gold Payment Recovery",
    silver_payment_metrics["settled_amount"],
    gold_recovery_metrics["settled_amount"],
)

add_reconciliation_result(
    "current_mrr",
    "Gold Customer 360",
    "Executive KPIs",
    gold_customer_metrics["current_mrr"],
    latest_executive_kpi_row["current_mrr"],
)

add_reconciliation_result(
    "current_arr",
    "Gold Customer 360",
    "Executive KPIs",
    gold_customer_metrics["current_arr"],
    latest_executive_kpi_row["current_arr"],
)

add_reconciliation_result(
    "collected_amount",
    "Gold Revenue Leakage",
    "Executive KPIs",
    gold_leakage_metrics["amount_paid"],
    latest_executive_kpi_row[
        "collected_amount"
    ],
)

add_reconciliation_result(
    "revenue_exposure_amount",
    "Gold Revenue Leakage",
    "Executive KPIs",
    gold_leakage_metrics[
        "total_revenue_exposure_amount"
    ],
    latest_executive_kpi_row[
        "total_revenue_exposure_amount"
    ],
)

add_reconciliation_result(
    "recovered_amount",
    "Gold Payment Recovery",
    "Executive KPIs",
    gold_recovery_metrics[
        "recovered_amount"
    ],
    latest_executive_kpi_row[
        "recovered_amount"
    ],
)

add_reconciliation_result(
    "unrecovered_amount",
    "Gold Payment Recovery",
    "Executive KPIs",
    gold_recovery_metrics[
        "unrecovered_amount"
    ],
    latest_executive_kpi_row[
        "unrecovered_amount"
    ],
)

add_reconciliation_result(
    "pending_collection_amount",
    "Gold Payment Recovery",
    "Executive KPIs",
    gold_recovery_metrics[
        "pending_collection_amount"
    ],
    latest_executive_kpi_row[
        "pending_collection_amount"
    ],
)


cross_layer_reconciliation_df = (
    spark.createDataFrame(
        reconciliation_results,
        [
            "metric_name",
            "source_layer",
            "target_layer",
            "source_value",
            "target_value",
            "difference",
            "test_status",
        ],
    )
)


failed_reconciliation_count = (
    cross_layer_reconciliation_df
    .where(
        F.col("test_status") != "PASS"
    )
    .count()
)


maximum_absolute_difference = (
    cross_layer_reconciliation_df
    .agg(
        F.max(
            F.abs("difference")
        ).alias(
            "maximum_absolute_difference"
        )
    )
    .first()[
        "maximum_absolute_difference"
    ]
)


assert failed_reconciliation_count == 0, (
    "One or more cross-layer reconciliations failed."
)

assert maximum_absolute_difference == 0.00, (
    "Cross-layer metrics contain a non-zero difference."
)


print(
    "Cross-layer reconciliation tests: "
    f"{len(reconciliation_results):,}"
)

print(
    "Failed reconciliation tests: "
    f"{failed_reconciliation_count:,}"
)

print(
    "Maximum absolute difference: "
    f"{maximum_absolute_difference:,.2f}"
)

print(
    "All Silver, Gold, and Executive metrics "
    "reconciled successfully."
)


display(
    cross_layer_reconciliation_df
    .orderBy(
        "metric_name",
        "source_layer",
    )
)

## 4. Validate Gold Auditability and Delta Merge Stability

Validate deterministic record hashes and inspect the latest Delta operation for every Gold model.

Each Gold table must contain valid SHA-256 hashes, preserve complete audit metadata, and show an idempotent latest MERGE with zero inserted, updated, or deleted records.

In [0]:
from delta.tables import DeltaTable


# Validate each Gold model using its actual audit-column contract.

GOLD_OPERATIONAL_CONFIG = [
    {
        "gold_model": "Customer 360",
        "table_name": (
            f"{CATALOG}.{GOLD_SCHEMA}.customer_360"
        ),
        "hash_column": "_gold_record_hash",
        "audit_timestamp_columns": [],
    },
    {
    "gold_model": "Revenue Leakage",
    "table_name": f"{CATALOG}.{GOLD_SCHEMA}.revenue_leakage",
    "hash_column": "_gold_record_hash",
    "audit_timestamp_columns": [
        "_silver_processed_at",
    ],
},
    {
        "gold_model": "Payment Recovery",
        "table_name": (
            f"{CATALOG}.{GOLD_SCHEMA}.payment_recovery"
        ),
        "hash_column": "_gold_record_hash",
        "audit_timestamp_columns": [],
    },
    {
        "gold_model": "Executive KPIs",
        "table_name": (
            f"{CATALOG}.{GOLD_SCHEMA}.executive_kpis"
        ),
        "hash_column": "_record_hash",
        "audit_timestamp_columns": [
            "_created_at",
            "_updated_at",
        ],
    },
]


gold_operational_results = []


for model_config in GOLD_OPERATIONAL_CONFIG:
    gold_model_name = (
        model_config["gold_model"]
    )

    gold_table_name = (
        model_config["table_name"]
    )

    hash_column = (
        model_config["hash_column"]
    )

    audit_timestamp_columns = (
        model_config[
            "audit_timestamp_columns"
        ]
    )

    gold_model_df = spark.table(
        gold_table_name
    )

    gold_model_columns = set(
        gold_model_df.columns
    )

    required_audit_columns = (
        [hash_column]
        + audit_timestamp_columns
    )

    missing_audit_columns = [
        column_name
        for column_name in required_audit_columns
        if column_name not in gold_model_columns
    ]

    invalid_hash_count = 0
    null_audit_field_count = 0

    if not missing_audit_columns:
        invalid_hash_count = (
            gold_model_df
            .where(
                F.col(hash_column).isNull()
                | (
                    F.length(
                        F.col(hash_column)
                    )
                    != 64
                )
            )
            .count()
        )

        null_audit_condition = (
            F.col(hash_column).isNull()
        )

        for audit_column in audit_timestamp_columns:
            null_audit_condition = (
                null_audit_condition
                | F.col(audit_column).isNull()
            )

        null_audit_field_count = (
            gold_model_df
            .where(
                null_audit_condition
            )
            .count()
        )

    gold_delta_table = (
        DeltaTable.forName(
            spark,
            gold_table_name,
        )
    )

    latest_history_row = (
        gold_delta_table
        .history(1)
        .first()
    )

    latest_operation = (
        latest_history_row["operation"]
    )

    latest_operation_metrics = (
        latest_history_row[
            "operationMetrics"
        ]
        or {}
    )

    inserted_record_count = int(
        latest_operation_metrics.get(
            "numTargetRowsInserted",
            0,
        )
    )

    updated_record_count = int(
        latest_operation_metrics.get(
            "numTargetRowsUpdated",
            0,
        )
    )

    deleted_record_count = int(
        latest_operation_metrics.get(
            "numTargetRowsDeleted",
            0,
        )
    )

    operational_test_status = (
        "PASS"
        if (
            not missing_audit_columns
            and invalid_hash_count == 0
            and null_audit_field_count == 0
            and latest_operation == "MERGE"
            and inserted_record_count == 0
            and updated_record_count == 0
            and deleted_record_count == 0
        )
        else "FAIL"
    )

    gold_operational_results.append(
        (
            gold_model_name,
            gold_table_name,
            hash_column,
            ",".join(
                audit_timestamp_columns
            ),
            ",".join(
                missing_audit_columns
            ),
            invalid_hash_count,
            null_audit_field_count,
            latest_operation,
            inserted_record_count,
            updated_record_count,
            deleted_record_count,
            operational_test_status,
        )
    )


gold_operational_results_df = (
    spark.createDataFrame(
        gold_operational_results,
        [
            "gold_model",
            "table_name",
            "hash_column",
            "audit_timestamp_columns",
            "missing_audit_columns",
            "invalid_hash_count",
            "null_audit_field_count",
            "latest_delta_operation",
            "latest_inserted_record_count",
            "latest_updated_record_count",
            "latest_deleted_record_count",
            "test_status",
        ],
    )
)


failed_gold_operational_test_count = (
    gold_operational_results_df
    .where(
        F.col("test_status") != "PASS"
    )
    .count()
)


total_invalid_hash_count = (
    gold_operational_results_df
    .agg(
        F.sum(
            "invalid_hash_count"
        ).alias(
            "total_invalid_hash_count"
        )
    )
    .first()[
        "total_invalid_hash_count"
    ]
)


total_null_audit_field_count = (
    gold_operational_results_df
    .agg(
        F.sum(
            "null_audit_field_count"
        ).alias(
            "total_null_audit_field_count"
        )
    )
    .first()[
        "total_null_audit_field_count"
    ]
)


total_changed_records_in_latest_merges = (
    gold_operational_results_df
    .agg(
        F.sum(
            F.col(
                "latest_inserted_record_count"
            )
            + F.col(
                "latest_updated_record_count"
            )
            + F.col(
                "latest_deleted_record_count"
            )
        ).alias(
            "total_changed_records"
        )
    )
    .first()[
        "total_changed_records"
    ]
)


assert failed_gold_operational_test_count == 0, (
    "One or more Gold operational tests failed."
)

assert total_invalid_hash_count == 0, (
    "Invalid Gold record hashes detected."
)

assert total_null_audit_field_count == 0, (
    "Null Gold audit fields detected."
)

assert total_changed_records_in_latest_merges == 0, (
    "The latest Gold MERGE operations changed records."
)


print(
    "Gold models tested: "
    f"{len(GOLD_OPERATIONAL_CONFIG):,}"
)

print(
    "Failed Gold operational tests: "
    f"{failed_gold_operational_test_count:,}"
)

print(
    "Invalid Gold record hashes: "
    f"{total_invalid_hash_count:,}"
)

print(
    "Null Gold audit fields: "
    f"{total_null_audit_field_count:,}"
)

print(
    "Records changed by latest Gold MERGE operations: "
    f"{total_changed_records_in_latest_merges:,}"
)

print(
    "All Gold models are auditable and "
    "operationally idempotent."
)


display(
    gold_operational_results_df
    .orderBy(
        "gold_model"
    )
)

## 5. Produce the Final Operational Test Summary

Aggregate the table inventory, business-key, reference-integrity, cross-layer reconciliation, and Gold operational results into one final end-to-end test summary.

The smoke test passes only when all 56 operational checks complete successfully.

In [0]:
from pyspark.sql import functions as F

# Combine all smoke-test categories into one final summary.

test_categories = [
    ("Table inventory and row-count validation", smoke_test_inventory_df),
    ("Business-key validation", key_integrity_results_df),
    ("Reference-integrity validation", reference_integrity_results_df),
    ("Cross-layer reconciliation", cross_layer_reconciliation_df),
    ("Gold auditability and idempotency", gold_operational_results_df),
]

smoke_test_summary_rows = []

for test_category, results_df in test_categories:
    tests_executed = results_df.count()

    if "test_status" in results_df.columns:
        status_column = "test_status"
    elif "overall_status" in results_df.columns:
        status_column = "overall_status"
    else:
        raise ValueError(
            f"No status column found for: {test_category}"
        )

    failed_tests = (
        results_df
        .filter(F.col(status_column) == "FAIL")
        .count()
    )

    passed_tests = tests_executed - failed_tests
    category_status = "PASS" if failed_tests == 0 else "FAIL"

    smoke_test_summary_rows.append(
        (
            test_category,
            tests_executed,
            passed_tests,
            failed_tests,
            category_status,
        )
    )

smoke_test_summary_df = spark.createDataFrame(
    smoke_test_summary_rows,
    [
        "test_category",
        "tests_executed",
        "passed_tests",
        "failed_tests",
        "test_status",
    ],
)

total_tests_executed = sum(row[1] for row in smoke_test_summary_rows)
total_passed_tests = sum(row[2] for row in smoke_test_summary_rows)
total_failed_tests = sum(row[3] for row in smoke_test_summary_rows)

EXPECTED_TOTAL_TEST_COUNT = 56

assert total_tests_executed == EXPECTED_TOTAL_TEST_COUNT, (
    f"Unexpected total smoke-test count: {total_tests_executed:,}"
)

assert total_failed_tests == 0, (
    "One or more end-to-end smoke tests failed."
)

print(f"Operational test categories: {len(smoke_test_summary_rows):,}")
print(f"Total tests executed: {total_tests_executed:,}")
print(f"Passed tests: {total_passed_tests:,}")
print(f"Failed tests: {total_failed_tests:,}")
print("Lakehouse layers validated: 3")
print("Delta tables validated: 12")
print("END-TO-END LAKEHOUSE SMOKE TEST PASSED.")

display(
    smoke_test_summary_df
    .orderBy("test_category")
)